# 🧠 Sentiment Analysis Model Training Notebook
This notebook handles the end-to-end Machine Learning training pipeline for our NLP dashboard. It loads the sample dataset, preprocesses the text data, trains a Multi-Class Logistic Regression pipeline, evaluates the results, and serializes the final model to disk.

In [3]:
import os
import sys
import pandas as pd
import numpy as np
%pip install nltk
import nltk
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

# Ensure notebook can look up modules in the parent src/ directory
sys.path.append(os.path.abspath(os.path.join('..')))
from src.preprocess import TextPreprocessor
from src.utils import save_model, ensure_directories

  Using cached nltk-3.9.4-py3-none-any.whl.metadata (3.2 kB)
Using cached nltk-3.9.4-py3-none-any.whl (1.6 MB)

   ---------------------------------------- 0/3 [regex]
   ---------------------------------------- 0/3 [regex]
   ------------- -------------------------- 1/3 [click]
   ------------- -------------------------- 1/3 [click]
   ------------- -------------------------- 1/3 [click]
   ------------- -------------------------- 1/3 [click]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
  

### 1. Ingest Dataset
Load the mock dataset generated for the application workspace.

In [4]:
df = pd.read_csv('../data/sample_reviews.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

Dataset Shape: (320, 5)


,review_id,review_text,rating,category,review_date
0,REV_0001,"It works okay, but nothing extraordinary. Stan...",3,Clothing,2026-03-04
1,REV_0002,"Extremely comfortable, fits perfectly and look...",4,Books,2026-06-01
2,REV_0003,Highly recommend to anyone looking for high qu...,4,Electronics,2026-03-01
3,REV_0004,"It works okay, but nothing extraordinary. Stan...",3,Electronics,2026-06-16
4,REV_0005,Customer service was completely unhelpful when...,2,Home & Kitchen,2026-05-31


### 2. Map Ground Truth Labels
We convert the ratings mapping directly to target labels:
- Ratings 1-2 $\rightarrow$ `0` (Negative)
- Rating 3 $\rightarrow$ `1` (Neutral)
- Ratings 4-5 $\rightarrow$ `2` (Positive)

In [5]:
def map_rating_to_label(rating):
    if rating in [1, 2]:
        return 0  # Negative
    elif rating == 3:
        return 1  # Neutral
    else:
        return 2  # Positive

df['target'] = df['rating'].apply(map_rating_to_label)
print("Target Class Distribution:")
print(df['target'].value_counts(normalize=True))

Target Class Distribution:
target
2    0.540625
0    0.293750
1    0.165625
Name: proportion, dtype: float64


### 3. Text Preprocessing
Leverage our centralized cleaning script module to standardize reviews corpus data strings.

In [6]:
preprocessor = TextPreprocessor()
df['cleaned_text'] = df['review_text'].apply(preprocessor.process_for_modeling)
df[['review_text', 'cleaned_text']].head()

,review_text,cleaned_text
0,"It works okay, but nothing extraordinary. Stan...",works okay nothing extraordinary standard perf...
1,"Extremely comfortable, fits perfectly and look...",extremely comfortable fits perfectly looks ama...
2,Highly recommend to anyone looking for high qu...,highly recommend anyone looking high quality g...
3,"It works okay, but nothing extraordinary. Stan...",works okay nothing extraordinary standard perf...
4,Customer service was completely unhelpful when...,customer service completely unhelpful asked re...


### 4. Split and Train Pipeline
Create a clean pipeline that wraps the TF-IDF feature extraction vectorizer alongside a multi-class Logistic Regression estimator.

In [7]:
X = df['cleaned_text']
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

sentiment_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=2500, ngram_range=(1, 2))),
    ('classifier', LogisticRegression(C=1.0, max_iter=1000, random_state=42))
])

print("Training Pipeline Model...")
sentiment_pipeline.fit(X_train, y_train)
print("Model training complete!")

Training Pipeline Model...
Model training complete!


### 5. Evaluate Pipeline Performance

In [8]:
y_pred = sentiment_pipeline.predict(X_test)
target_names = ['Negative', 'Neutral', 'Positive']

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=target_names))

print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))


--- Classification Report ---
              precision    recall  f1-score   support

    Negative       1.00      1.00      1.00        19
     Neutral       1.00      1.00      1.00        11
    Positive       1.00      1.00      1.00        34

    accuracy                           1.00        64
   macro avg       1.00      1.00      1.00        64
weighted avg       1.00      1.00      1.00        64

--- Confusion Matrix ---
[[19  0  0]
 [ 0 11  0]
 [ 0  0 34]]


### 6. Export Trained Model Artifact
Serialize the trained model pipeline to `models/sentiment_model.pkl` so the Streamlit frontend can use it directly.

In [11]:
ensure_directories()
model_output_path = "D:/NLP_DASHBOARD/models/sentiment_model.pkl"

save_model(sentiment_pipeline, model_output_path)
print(f"🎯 Successfully exported trained model pipeline package to: {model_output_path}")

🎯 Successfully exported trained model pipeline package to: D:/NLP_DASHBOARD/models/sentiment_model.pkl
